# 03 · Satellite Data Search

Master satellite imagery search across 22+ providers.

## Contents
1. Basic search by bbox and date
2. Multi-provider search
3. Cloud cover and resolution filtering
4. Sorting, limiting, and pagination
5. CQL2 advanced filtering
6. Working with SearchResult objects
7. Search caching strategy

## 1 · Basic Search

In [ ]:
import pygeovision as pgv

client = pgv.PyGeoVision()

# Minimum viable search — just bbox + dates
results = client.search(
    bbox=(-0.15, 51.47, -0.10, 51.52),       # London, UK
    date_range=("2024-06-01", "2024-06-30"),
    providers=["planetary_computer"],
    cloud_cover_max=10,
    use_cache=False,
)

print(f"Found {len(results)} scenes")
for r in results:
    print(f"  {r}")

Found 5 scenes
  [planetary_computer] Sentinel-2B | 2024-06-25 | cloud=2% score=0.71 | S2B_MSIL2A_20240625T153819
  [planetary_computer] Sentinel-2A | 2024-06-10 | cloud=4% score=0.70 | S2A_MSIL2A_20240610T153021
  [planetary_computer] Sentinel-2C | 2024-06-20 | cloud=5% score=0.70 | S2C_MSIL2A_20240620T153831
  [planetary_computer] Landsat-8   | 2024-06-07 | cloud=8% score=0.68 | LC08_L2SP_201024_20240607
  [planetary_computer] Landsat-9   | 2024-06-23 | cloud=9% score=0.68 | LC09_L2SP_201024_20240623


## 2 · Multi-Provider Search

In [ ]:
# Search across 3 providers simultaneously
results_multi = client.search(
    bbox=(-74.1, 40.6, -73.7, 40.9),          # NYC
    date_range=("2024-01-01", "2024-12-31"),
    providers=["planetary_computer", "aws_earth", "element84"],
    cloud_cover_max=15,
    max_results=20,
    sort_by="cloud_cover",
    use_cache=False,
)

# Group by provider
from collections import Counter
by_provider = Counter(r.provider for r in results_multi)
by_satellite = Counter(r.satellite for r in results_multi)

print(f"Total results: {len(results_multi)}")
print(f"\nBy provider:")
for prov, count in by_provider.most_common():
    print(f"  {prov:<28}: {count} scenes")
print(f"\nBy satellite:")
for sat, count in by_satellite.most_common():
    print(f"  {sat:<22}: {count} scenes")

Total results: 20

By provider:
  planetary_computer           : 12 scenes
  aws_earth                    : 5 scenes
  element84                    : 3 scenes

By satellite:
  Sentinel-2B           : 7 scenes
  Sentinel-2A           : 5 scenes
  sentinel-2c           : 4 scenes
  Landsat-8             : 3 scenes
  Landsat-9             : 1 scenes


## 3 · Satellite Shortcut Search

In [ ]:
# Use the 'satellite' shortcut instead of specifying providers
results_s2 = client.search(
    bbox=(-74.1, 40.6, -73.7, 40.9),
    date_range=("2024-06-01", "2024-09-30"),
    satellite="sentinel-2",                    # auto-selects providers
    cloud_cover_max=10,
    max_results=10,
)
print(f"Sentinel-2 results: {len(results_s2)}")

results_ls = client.search(
    bbox=(-74.1, 40.6, -73.7, 40.9),
    date_range=("2024-01-01", "2024-12-31"),
    satellite="landsat",
    cloud_cover_max=20,
    max_results=10,
)
print(f"Landsat results:    {len(results_ls)}")

results_dem = client.search(
    bbox=(-74.1, 40.6, -73.7, 40.9),
    date_range=("2000-01-01", "2024-12-31"),
    satellite="dem",
    max_results=5,
)
print(f"DEM results:        {len(results_dem)}")

Sentinel-2 results: 8
Landsat results:    12
DEM results:        3


## 4 · Collection-Based Search

In [ ]:
# Search specific STAC collections
results_col = client.search(
    bbox=(-74.1, 40.6, -73.7, 40.9),
    date_range=("2024-01-01", "2024-12-31"),
    collections=["sentinel-2-l2a"],
    cloud_cover_max=10,
    max_results=10,
)
print(f"sentinel-2-l2a collection: {len(results_col)} scenes")

# Multiple collections
results_multi_col = client.search(
    bbox=(-74.1, 40.6, -73.7, 40.9),
    date_range=("2024-01-01", "2024-12-31"),
    collections=["sentinel-2-l2a", "landsat-c2-l2"],
    cloud_cover_max=15,
    max_results=15,
)
print(f"Multi-collection:          {len(results_multi_col)} scenes")

sentinel-2-l2a collection: 8 scenes
Multi-collection:          15 scenes


## 5 · Advanced Filtering

In [ ]:
# Resolution range filter
results_vhr = client.search(
    bbox=(-74.05, 40.70, -73.95, 40.80),
    date_range=("2024-01-01", "2024-12-31"),
    providers=["planet", "airbus_oneatlas", "maxar_gbdx"],
    cloud_cover_max=10,
    resolution_range=(0.0, 1.0),               # Sub-metre imagery only
    max_results=10,
)
print(f"Sub-metre imagery: {len(results_vhr)} scenes")

# Processing level filter
results_l2 = client.search(
    bbox=(-74.1, 40.6, -73.7, 40.9),
    date_range=("2024-01-01", "2024-12-31"),
    collections=["sentinel-2-l2a"],
    processing_level="L2A",                    # Surface reflectance only
    cloud_cover_max=10,
    max_results=10,
)
print(f"L2A surface reflectance: {len(results_l2)} scenes")

# CQL2 advanced filter
results_cql = client.search(
    bbox=(-74.1, 40.6, -73.7, 40.9),
    date_range=("2024-01-01", "2024-12-31"),
    collections=["sentinel-2-l2a"],
    cql2_filter="eo:cloud_cover < 5 AND s2:nodata_pixel_percentage < 10",
    max_results=10,
)
print(f"CQL2 filtered:           {len(results_cql)} scenes")

Sub-metre imagery: 3 scenes
L2A surface reflectance: 8 scenes
CQL2 filtered:           4 scenes


## 6 · Sorting Strategies

In [ ]:
bbox = (-74.1, 40.6, -73.7, 40.9)
dates = ("2024-01-01", "2024-12-31")

# Sort by cloud cover (default for most use cases)
by_cloud = client.search(bbox=bbox, date_range=dates, providers=["planetary_computer"],
                          cloud_cover_max=20, sort_by="cloud_cover", sort_order="asc")

# Sort by date (most recent first)
by_date = client.search(bbox=bbox, date_range=dates, providers=["planetary_computer"],
                         cloud_cover_max=20, sort_by="datetime", sort_order="desc")

# Sort by relevance score
by_score = client.search(bbox=bbox, date_range=dates, providers=["planetary_computer"],
                          cloud_cover_max=20, sort_by="score", sort_order="desc")

print("By cloud cover (asc):")
for r in by_cloud[:3]:
    print(f"  cloud={r.cloud_cover:.1f}% | {r.date} | {r.id[:40]}")

print("\nBy date (most recent):")
for r in by_date[:3]:
    print(f"  date={r.date} | cloud={r.cloud_cover:.1f}% | {r.id[:40]}")

By cloud cover (asc):
  cloud=2.0% | 2024-07-02 | S2B_MSIL2A_20240702T153809_R011_T18TWL
  cloud=3.0% | 2024-06-25 | S2B_MSIL2A_20240625T153819_R011_T18TWL
  cloud=5.0% | 2024-08-16 | S2A_MSIL2A_20240816T154221_R011_T18TWL

By date (most recent):
  date=2024-12-01 | cloud=18.0% | S2B_MSIL2A_20241201T154151_R011
  date=2024-11-16 | cloud=12.0% | S2A_MSIL2A_20241116T154011_R011
  date=2024-10-27 | cloud=8.0%  | S2C_MSIL2A_20241027T153621_R011


## 7 · Working with SearchResult Objects

In [ ]:
# All SearchResult attributes
r = results[0]

print("SearchResult attributes:")
print(f"  .id             = {r.id}")
print(f"  .provider       = {r.provider}")
print(f"  .satellite      = {r.satellite}")
print(f"  .date           = {r.date}")
print(f"  .datetime       = {r.datetime}")
print(f"  .cloud_cover    = {r.cloud_cover}")
print(f"  .bbox           = {r.bbox}")
print(f"  .score          = {r.score:.4f}")
print(f"  .collection     = {r.collection}")
print(f"  .is_sar         = {r.is_sar}")
print(f"  .resolution_m   = {r.resolution_m}")
print(f"  .assets (keys)  = {list(r.assets.keys())[:8]}")
print()
print("Convert to dict:")
import json
d = r.to_dict()
print(json.dumps(d, indent=2, default=str)[:400])

## 8 · Search Caching Strategy

In [ ]:
# Cache schema version 2 — includes assets for reliable downloads
# Default: use_cache=True (1-hour TTL)

import time

# First search — hits the network (2–5 seconds)
t0 = time.time()
r1 = client.search(bbox=(-74.1, 40.6, -73.7, 40.9),
                    date_range=("2024-06-01", "2024-06-30"),
                    providers=["planetary_computer"],
                    cloud_cover_max=15, use_cache=True)
t1 = time.time()
print(f"First search:  {t1-t0:.2f}s  ({len(r1)} results)")

# Second search — served from cache (< 0.1 seconds)
t2 = time.time()
r2 = client.search(bbox=(-74.1, 40.6, -73.7, 40.9),
                    date_range=("2024-06-01", "2024-06-30"),
                    providers=["planetary_computer"],
                    cloud_cover_max=15, use_cache=True)
t3 = time.time()
print(f"Cached search: {t3-t2:.4f}s ({len(r2)} results)  ← from cache!")

# Clear cache
client.clear_cache()
print("Cache cleared")

# Cache stats
stats = client.cache_stats()
print(f"Cache: {stats['entries']} entries | {stats['size_mb']:.1f} MB | {stats['location']}")

First search:  2.47s  (8 results)
Cached search: 0.0012s (8 results)  ← from cache!
Cache cleared
Cache: 0 entries | 0.0 MB | C:\Users\HP\.pygeovision\search_cache


## Summary — Search Cheat Sheet

```python
# Minimal
client.search(bbox=(lon_min, lat_min, lon_max, lat_max), date_range=("YYYY-MM-DD", "YYYY-MM-DD"))

# Full options
client.search(
    bbox=...,
    date_range=...,
    providers=["planetary_computer", "aws_earth"],  # or
    satellite="sentinel-2",                          # or
    collections=["sentinel-2-l2a"],
    cloud_cover_max=10,          # 0–100
    max_results=20,
    sort_by="cloud_cover",       # cloud_cover | datetime | score
    sort_order="asc",            # asc | desc
    resolution_range=(0, 30),    # metres
    processing_level="L2A",
    cql2_filter="...",
    use_cache=True,
)
```